In [1]:
#Core Libraries
import os
import glob
import sys
import torch
import cv2
import random
import zipfile
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.model_selection import train_test_split

#PyTorch Libraries
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2

BASE_DIR = r"C:\Users\KEITH FERNANDES\OneDrive\Desktop\MAMMOGRAPHY\CBIS-DDSM"
PAIR_CSV = f"{BASE_DIR}/segmentation_pairs.csv"

df = pd.read_csv(PAIR_CSV)
print("Total segmentation samples:", len(df))

Total segmentation samples: 3463


In [5]:
def setup_and_download_data():
    """
    Handles Kaggle API setup, library installation, data download, and unzipping.
    """
    print("--- Step 1 of 7: Configuring Kaggle API ---")
    
    # --- PASTE YOUR KAGGLE CREDENTIALS HERE ---
    kaggle_username = "keithfernandes14"
    kaggle_key = "KGAT_532ab66ee0ed9a3e3bf5bc9b2640f326"
    
    if "your-kaggle" in kaggle_username or "your-kaggle" in kaggle_key:
        print("\nERROR: Please open the script and replace the placeholder Kaggle credentials.")
        sys.exit()
        
    os.environ['KAGGLE_USERNAME'] = kaggle_username
    os.environ['KAGGLE_KEY'] = kaggle_key
    print("Kaggle API configured successfully using environment variables.")

    print("\n--- Step 2 of 7: Installing Kaggle Libraries ---")
    os.system('pip install kaggle kagglehub --quiet')
    import kagglehub
    print("Libraries installed.")
    print("\n--- Step 3 of 7: Downloading and Unzipping Dataset ---")

try:
    # DATASET YOU WANT
    DATASET = "awsaf49/cbis-ddsm-breast-cancer-image-dataset"

    # WHERE YOU WANT IT
    extract_path = "./Mammography"

    # Download dataset
    download_path = kagglehub.dataset_download(DATASET)
    print(f"Dataset downloaded to: {download_path}")

    # Handle zip vs directory
    if download_path.endswith(".zip"):
        os.makedirs(extract_path, exist_ok=True)
        with zipfile.ZipFile(download_path, "r") as zip_ref:
            zip_ref.extractall(extract_path)
        print(f"Dataset unzipped to '{extract_path}'")
    else:
        if not os.path.exists(extract_path):
            shutil.copytree(download_path, extract_path)
            print(f"Dataset copied to '{extract_path}'")
        else:
            print(f"Dataset already exists at '{extract_path}'")

except Exception as e:
    print(f"Error handling dataset: {e}")
    print("Please check your Kaggle credentials and try again.")
    sys.exit(1)

print("\n✅ Dataset setup complete.")


100%|██████████| 4.95G/4.95G [21:14<00:00, 4.17MB/s]  

Extracting files...


Dataset downloaded to: C:\Users\KEITH FERNANDES\.cache\kagglehub\datasets\awsaf49\cbis-ddsm-breast-cancer-image-dataset\versions\1
Dataset copied to './Mammography'

✅ Dataset setup complete.


In [2]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.double_conv(x)

class Down(nn.Module):
    def __init__(self, in_channel, out_channel):
        super().__init__()
        

In [3]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.double_conv(x)

class Down(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.maxpool_conv = nn.Sequential(nn.MaxPool2d(2), DoubleConv(in_channels, out_channels))
    def forward(self, x):
        return self.maxpool_conv(x)

class Up(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
        self.conv = DoubleConv(in_channels, out_channels)
    def forward(self, x1, x2):
        x1 = self.up(x1)
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)

class OutConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(OutConv, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=1)
    def forward(self, x):
        return self.conv(x)

class UNet(nn.Module):
    def __init__(self, n_channels, n_classes):
        super(UNet, self).__init__()
        self.inc = DoubleConv(n_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        self.down4 = Down(512, 1024)
        self.up1 = Up(1024, 512)
        self.up2 = Up(512, 256)
        self.up3 = Up(256, 128)
        self.up4 = Up(128, 64)
        self.outc = OutConv(64, n_classes)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        return self.outc(x)

In [12]:
class MammoSegDataset(Dataset):
    def __init__(self, df):
        """
        df must contain columns:
        - image_path
        - mask_path
        """
        self.df = df.reset_index(drop=True)

        if len(self.df) == 0:
            raise RuntimeError("Dataset is empty")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path  = self.df.loc[idx, "image_path"]
        mask_path = self.df.loc[idx, "mask_path"]

        image = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        mask  = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

        if image is None:
            raise FileNotFoundError(f"Image not found: {img_path}")
        if mask is None:
            raise FileNotFoundError(f"Mask not found: {mask_path}")

        # Resize (must match)
        image = cv2.resize(image, (256, 256))
        mask  = cv2.resize(mask,  (256, 256))

        # Normalize
        image = image.astype("float32") / 255.0
        mask  = (mask > 0).astype("float32")  # binary

        # To tensor (C,H,W)
        image = torch.tensor(image).unsqueeze(0)
        mask  = torch.tensor(mask).unsqueeze(0)

        return image, mask

from tqdm.auto import tqdm

def train_one_epoch(loader, model, optimizer, loss_fn, device):
    model.train()
    running_loss = 0.0

    total_batches = len(loader)

    for i, (images, masks) in enumerate(loader):
        images = images.to(device)
        masks  = masks.to(device)

        preds = model(images)
        loss  = loss_fn(preds, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        # ---- FORCE visible progress ----
        if i % 10 == 0:
            print(f"Batch {i}/{total_batches} | Loss: {loss.item():.4f}")

    return running_loss / total_batches


def validate_one_epoch(loader, model, loss_fn, device, smooth=1e-6):
    model.eval()
    val_loss = 0.0
    intersection, union = 0.0, 0.0

    with torch.no_grad():
        for images, masks in loader:
            images = images.to(device)
            masks  = masks.to(device)

            preds = model(images)
            val_loss += loss_fn(preds, masks).item()

            preds_bin = (torch.sigmoid(preds) > 0.5).float()
            intersection += (preds_bin * masks).sum()
            union += preds_bin.sum() + masks.sum()

    dice = (2 * intersection + smooth) / (union + smooth)
    return val_loss / len(loader), dice.item()

def evaluate_model(loader, model, device, smooth=1e-6):
    model.eval()
    tp = fp = fn = tn = 0.0

    with torch.no_grad():
        for images, masks in tqdm(loader, desc="Evaluating"):
            images = images.to(device)
            masks  = masks.to(device)

            preds = (torch.sigmoid(model(images)) > 0.5).float()

            tp += (preds * masks).sum().item()
            fp += (preds * (1 - masks)).sum().item()
            fn += ((1 - preds) * masks).sum().item()
            tn += ((1 - preds) * (1 - masks)).sum().item()

    accuracy = (tp + tn + smooth) / (tp + tn + fp + fn + smooth)
    precision = (tp + smooth) / (tp + fp + smooth)
    recall = (tp + smooth) / (tp + fn + smooth)
    dice = (2 * tp + smooth) / (2 * tp + fp + fn + smooth)
    iou = (tp + smooth) / (tp + fp + fn + smooth)

    print("\n--- Evaluation Results ---")
    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"Dice     : {dice:.4f}")
    print(f"IoU      : {iou:.4f}")
    print("--------------------------")

    return accuracy, precision, recall, dice, iou

def plot_training_curves(history):
    plt.figure(figsize=(14,5))

    plt.subplot(1,2,1)
    plt.plot(history["train_loss"], label="Train Loss")
    plt.plot(history["val_loss"], label="Val Loss")
    plt.legend(); plt.grid(); plt.title("Loss")

    plt.subplot(1,2,2)
    plt.plot(history["val_dice"], label="Val Dice", color="green")
    plt.legend(); plt.grid(); plt.title("Dice Score")

    plt.show()

def visualize_predictions(model, dataset, device, num_images=5):
    model.eval()
    plt.figure(figsize=(12, num_images * 4))

    for i in range(num_images):
        idx = random.randint(0, len(dataset) - 1)
        image, mask = dataset[idx]

        with torch.no_grad():
            pred = (torch.sigmoid(model(image.unsqueeze(0).to(device))) > 0.5).float()

        plt.subplot(num_images, 3, i*3+1)
        plt.imshow(image.squeeze(), cmap="gray")
        plt.title("Image"); plt.axis("off")

        plt.subplot(num_images, 3, i*3+2)
        plt.imshow(mask.squeeze(), cmap="gray")
        plt.title("GT Mask"); plt.axis("off")

        plt.subplot(num_images, 3, i*3+3)
        plt.imshow(pred.cpu().squeeze(), cmap="gray")
        plt.title("Predicted"); plt.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
def main():
    import os
    import torch
    import torch.nn as nn
    import pandas as pd
    from torch.utils.data import DataLoader
    from sklearn.model_selection import train_test_split

    # -----------------------------
    # Configuration
    # -----------------------------
    BASE_DIR = r"C:\Users\KEITH FERNANDES\OneDrive\Desktop\MAMMOGRAPHY\CBIS-DDSM"
    PAIR_CSV = os.path.join(BASE_DIR, "segmentation_pairs.csv")
    MODEL_PATH = "Mammography_UNet.pth"

    LEARNING_RATE = 1e-4
    BATCH_SIZE = 4        # segmentation is memory-heavy
    NUM_EPOCHS = 50
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

    torch.manual_seed(42)

    print(f"Using device: {DEVICE}")
    print("Training task: Mammography Segmentation (U-Net)")

    # -----------------------------
    # Step 1: Load segmentation pairs
    # -----------------------------
    if not os.path.exists(PAIR_CSV):
        raise FileNotFoundError(
            "segmentation_pairs.csv not found. "
            "You must generate image–mask pairs first."
        )

    df = pd.read_csv(PAIR_CSV)
    print("Total segmentation samples:", len(df))

    # -----------------------------
    # Step 2: Train / Val / Test split
    # -----------------------------
    df_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
    df_train, df_val  = train_test_split(df_train, test_size=0.25, random_state=42)

    print(f"Train: {len(df_train)} | Val: {len(df_val)} | Test: {len(df_test)}")

    # -----------------------------
    # Step 3: Datasets
    # -----------------------------
    train_ds = MammoSegDataset(df_train)
    val_ds   = MammoSegDataset(df_val)
    test_ds  = MammoSegDataset(df_test)

    # -----------------------------
    # Step 4: DataLoaders
    # -----------------------------
    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0
    )

    val_loader = DataLoader(
        val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2
    )

    test_loader = DataLoader(
        test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2
    )

    # -----------------------------
    # Step 5: Model, Loss, Optimizer
    # -----------------------------
    model = UNet(n_channels=1, n_classes=1).to(DEVICE)

    loss_fn = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

    # -----------------------------
    # Step 6: Training loop
    # -----------------------------
    print("\n--- Starting Training ---")

    history = {"train_loss": [], "val_loss": [], "val_dice": []}
    best_val_dice = 0.0

    for epoch in range(NUM_EPOCHS):
        train_loss = train_one_epoch(
            train_loader, model, optimizer, loss_fn, DEVICE
        )

        val_loss, val_dice = validate_one_epoch(
            val_loader, model, loss_fn, DEVICE
        )

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_dice"].append(val_dice)

        print(
            f"Epoch [{epoch+1}/{NUM_EPOCHS}] | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Dice: {val_dice:.4f}"
        )

        if val_dice > best_val_dice:
            best_val_dice = val_dice
            torch.save(model.state_dict(), MODEL_PATH)
            print(f"==> Best model saved (Dice: {best_val_dice:.4f})")

    # -----------------------------
    # Step 7: Final Evaluation
    # -----------------------------
    print("\n--- Final Evaluation on Test Set ---")
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))

    evaluate_model(test_loader, model, DEVICE)

    plot_training_curves(history)
    visualize_predictions(model, test_ds, DEVICE, num_images=10)


if __name__ == "__main__":
    main()

Using device: cpu
Training task: Mammography Segmentation (U-Net)
Total segmentation samples: 3463
Train: 2077 | Val: 693 | Test: 693

--- Starting Training ---
Batch 0/520 | Loss: 0.7131
Batch 10/520 | Loss: 0.3559
Batch 20/520 | Loss: 0.3353
Batch 30/520 | Loss: 0.7844
Batch 40/520 | Loss: 0.5953
Batch 50/520 | Loss: 0.5652
Batch 60/520 | Loss: 0.2624
Batch 70/520 | Loss: 0.2696
Batch 80/520 | Loss: 0.4903
Batch 90/520 | Loss: 0.2840
Batch 100/520 | Loss: 0.2757
Batch 110/520 | Loss: 0.4767
Batch 120/520 | Loss: 0.4648
Batch 130/520 | Loss: 0.2696
Batch 140/520 | Loss: 0.4811
Batch 150/520 | Loss: 0.3833
Batch 160/520 | Loss: 0.3639
Batch 170/520 | Loss: 0.2160
Batch 180/520 | Loss: 0.2175
Batch 190/520 | Loss: 0.3544
Batch 200/520 | Loss: 0.1771
Batch 210/520 | Loss: 0.3252
Batch 220/520 | Loss: 0.1838
Batch 230/520 | Loss: 0.3004
Batch 240/520 | Loss: 0.1851
Batch 250/520 | Loss: 0.1584
Batch 260/520 | Loss: 0.2979
Batch 270/520 | Loss: 0.1669
Batch 280/520 | Loss: 0.1513
Batch 290